# Logistic Regression — Chapter 6 (Layout templates)

Baseline **dual-panel 16:9** layout — no formula rails. Left panel is always the 2D roster with **w_ST / w_EL / b** knobs.

| Module | Purpose |
|--------|---------|
| `ch4_layout.py` | Knob assets, shared 16:9 constants |
| `ch6_layout.py` | Duo figure factory + right-panel drawers + datasets |

Canvas: **≈16.89×9.5 in @ 200 DPI** (same height as Chapter 4, full 16:9 width).

## Template groups

1. **Calibration** — reliability diagram, probability histogram, calibration gap
2. **ROC** — standard curve, threshold markers, model comparison
3. **Class imbalance** — class counts, confusion matrix, precision–recall

Run **`ch6_setup`**, then **`ch6-layout-previews`** to write PNGs under `renders/`.


In [ ]:
# --- Ch5 setup: layout + Ch3 builders + dual-panel frame helpers ---

import importlib
import json
from pathlib import Path

import ch4_layout
import ch6_layout

importlib.reload(ch4_layout)
importlib.reload(ch6_layout)
from ch4_layout import *
from ch6_layout import *

_CH3_NB = Path("logistic-regression-chap3.ipynb")
if not _CH3_NB.is_file():
    raise FileNotFoundError(_CH3_NB.resolve())
_ch3_src = "".join(json.loads(_CH3_NB.read_text())["cells"][1]["source"])
exec(compile(_ch3_src, str(_CH3_NB), "exec"), globals())
del _CH3_NB, _ch3_src


def ch6_predict_proba(ws, we, bb, study, exam):
    return sigmoid(logits_plane(ws, we, bb, study, exam))


def ch6_align_knobs(fig, ax_data, axes_k):
    """Align knob strip under the tuned 2D panel."""
    _ch3_align_knob_axes_under_data(fig, ax_data, axes_k)
    ch3_layout_knob_axes_like_bridge_end(fig, ax_data, axes_k)
    ch4_duo_knob_layout_tune(fig, ax_data, axes_k, height_scale=CH4_DUO_KNOB_HEIGHT_SCALE)



def ch6_frame_duo(
    *,
    study,
    exam,
    y,
    ws,
    we,
    bb,
    p,
    right_draw,
    emphasize_knob="st",
    show_colormap=True,
    right_kwargs=None,
):
    """Dual-panel 16:9 frame: left 2D+knobs, right topic panel."""
    right_kwargs = {} if right_kwargs is None else dict(right_kwargs)
    fig, ax_data, ax_right, axes_k = ch6_figure_duo()
    leg = legend_linear_equation_values_bold_param(float(ws), float(we), float(bb), None)
    ch3_draw_left_panel(
        ax_data, ws, we, bb, study, exam, y, leg,
        show_colormap=show_colormap, highlight_mistakes_flag=True,
    )
    knob_rgbs, canvas_sides = ch4_knob_asset_pack()
    ch3_draw_knob_row(
        fig, axes_k, ws, we, bb, emphasize_knob, knob_rgbs, canvas_sides, ax_data=ax_data,
    )
    ch6_align_knobs(fig, ax_data, axes_k)
    ch6_duo_right_panel_extent(fig, ax_data, ax_right, axes_k)
    right_draw(ax_right, y, p, **right_kwargs)
    return fig_to_image(fig, dpi=CH6_EXPORT_DPI)


def ch6_export_layout_previews():
    """Write all baseline layout templates to renders/."""
    specs = [
        ("ch6_calib_reliability_preview.png", CH6_CALIBRATION_POINTS, CH6_WEIGHTS_CALIBRATION, ch6_draw_calibration_reliability, {}),
        ("ch6_calib_histogram_preview.png", CH6_CALIBRATION_POINTS, CH6_WEIGHTS_CALIBRATION, ch6_draw_calibration_histogram, {}),
        ("ch6_calib_gap_preview.png", CH6_CALIBRATION_POINTS, CH6_WEIGHTS_CALIBRATION, ch6_draw_calibration_gap, {}),
        ("ch6_roc_standard_preview.png", CH6_ROC_POINTS, CH6_WEIGHTS_ROC, ch6_draw_roc_standard, {}),
        ("ch6_roc_thresholds_preview.png", CH6_ROC_POINTS, CH6_WEIGHTS_ROC, ch6_draw_roc_thresholds, {}),
        ("ch6_imbal_bars_preview.png", CH6_IMBALANCED_POINTS, CH6_WEIGHTS_IMBALANCED, ch6_draw_imbalance_bars, {}),
        ("ch6_imbal_confusion_preview.png", CH6_IMBALANCED_POINTS, CH6_WEIGHTS_IMBALANCED, ch6_draw_imbalance_confusion, {}),
        ("ch6_imbal_pr_preview.png", CH6_IMBALANCED_POINTS, CH6_WEIGHTS_IMBALANCED, ch6_draw_imbalance_pr, {}),
    ]
    paths = []
    for fn, pts, w, draw, kw in specs:
        st, ex, y = ch6_unpack_points(pts)
        ws, we, bb = w
        p = ch6_predict_proba(ws, we, bb, st, ex)
        img = ch6_frame_duo(study=st, exam=ex, y=y, ws=ws, we=we, bb=bb, p=p, right_draw=draw, right_kwargs=kw)
        paths.append(ch6_save_preview(img, fn))
    # ROC compare needs two probability vectors
    st, ex, y = ch6_unpack_points(CH6_ROC_POINTS)
    ws, we, bb = CH6_WEIGHTS_ROC
    p_main = ch6_predict_proba(ws, we, bb, st, ex)
    ws2, we2, bb2 = CH6_WEIGHTS_ROC_ALT
    p_alt = ch6_predict_proba(ws2, we2, bb2, st, ex)
    img = ch6_frame_duo(
        study=st, exam=ex, y=y, ws=ws, we=we, bb=bb, p=p_main,
        right_draw=ch6_draw_roc_compare,
        right_kwargs={"p_alt": p_alt},
    )
    paths.append(ch6_save_preview(img, "ch6_roc_compare_preview.png"))
    for p in paths:
        print("wrote", p)
    return paths


print("Chapter 6 layout OK — canvas", CH6_FIGSIZE, "@", CH6_EXPORT_DPI, "DPI")


### Calibration templates

Dataset: noisy roster (`CH6_CALIBRATION_POINTS`, 26 students with boundary flips).


In [ ]:
ch6_export_layout_previews()
